# Модель 2: rubert-tiny2

Дообучаем небольшую русскоязычную нейросеть [cointegrated/rubert-tiny2](https://huggingface.co/cointegrated/rubert-tiny2) и сравниваем её с моделью 1 (TF-IDF + логистическая регрессия) **на одних и тех же данных**. Таблица сравнения в конце ноутбука — главный результат для отчёта.

Цели: F1-macro ≥ 0.80 по тональности, ≥ 0.75 по категориям, 1000 отзывов меньше чем за минуту.

**Перед запуском в Colab:** Среда выполнения → Сменить среду выполнения → **T4 GPU**. На процессоре всё работает, но обучение займёт в десятки раз больше времени.

Первая ячейка нужна только в Colab: она скачивает проект с GitHub. Папка `ml/` появится там после того, как команда отправит изменения в репозиторий. До этого можно загрузить папку `ml` в Colab вручную (значок папки слева → загрузить).

In [ ]:
import sys, os
if "google.colab" in sys.modules and not os.path.exists("ai-customer-insight"):
    !git clone https://github.com/3092334al-del/ai-customer-insight.git
    %cd ai-customer-insight/ml/notebooks
    !pip install -q -r ../requirements-bert.txt

In [ ]:
import sys, time
from pathlib import Path

ML = Path.cwd().parent
sys.path.insert(0, str(ML))

import pandas as pd
import matplotlib.pyplot as plt
from aci_ml import baseline, bert
from aci_ml.dataset import load_dataset

device = bert.pick_device()
print("Устройство:", device)
if device == "cpu":
    print("Внимание: GPU не найден, обучение будет медленным")

## 1. Данные

Те же файлы, что в ноутбуке 01. Данные делятся на три части:
* **обучение** — на них модель учится;
* **валидация** (10 % от обучающих) — по ней выбирается лучшая эпоха;
* **тест** — модель видит их один раз, в самом конце. Только эти цифры можно писать в отчёт.

In [ ]:
TRAIN = ML / "data" / "raw" / "train.xml"
TEST = ML / "data" / "raw" / "test.xml"
OWN = []  # например: [ML / "data" / "наша_разметка.csv"]
LEVEL = "sentence"

if TRAIN.exists():
    train_df = pd.concat([load_dataset(TRAIN, LEVEL)] + [load_dataset(p) for p in OWN], ignore_index=True)
    test_df = load_dataset(TEST, LEVEL) if TEST.exists() else None
else:
    print("Корпуса нет — используем тестовый пример, цифры ничего не значат")
    sample = pd.concat([load_dataset(ML / "tests" / "fixtures" / "semeval_ru_sample.xml"),
                        load_dataset(ML / "data" / "разметка_шаблон.csv")], ignore_index=True)
    train_df, test_df = pd.concat([sample] * 4, ignore_index=True), sample

if test_df is None:
    train_df, test_df = baseline.split(train_df)
train_df, val_df = baseline.split(train_df, test_size=0.1)
print(f"Обучение: {len(train_df)}, валидация: {len(val_df)}, тест: {len(test_df)}")

## 2. Настройки

С этими числами стоит поэкспериментировать и записать результаты в таблицу для отчёта:
* `EPOCHS` — сколько раз модель просматривает все данные (3–10);
* `LR` — скорость обучения (попробуйте 3e-5, 5e-5, 1e-4);
* `MAX_LENGTH` — сколько токенов отзыва учитывается; для целых отзывов (`LEVEL = "review"`) поставьте 512.

In [ ]:
EPOCHS = 5
LR = 5e-5
BATCH_SIZE = 32
MAX_LENGTH = 256

## 3. Модель 1 на тех же данных — для сравнения

In [ ]:
model1 = baseline.train(pd.concat([train_df, val_df]))
metrics1 = baseline.evaluate(model1, test_df)
print(baseline.format_report(metrics1))

## 4. Обучение rubert-tiny2

Предупреждение «Some weights … were not initialized» — нормально: это новый слой-классификатор поверх нейросети, его мы и обучаем.

In [ ]:
start = time.time()
model2 = bert.train(train_df, val_df, epochs=EPOCHS, lr=LR, batch_size=BATCH_SIZE,
                    max_length=MAX_LENGTH, device=device)
print(f"Обучение заняло {time.time() - start:.0f} с")

Графики обучения. Если потери падают, а F1 на валидации перестал расти или снижается — модель начала переобучаться; больше эпох не нужно.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
for ax, (task, title) in zip(axes, [("sentiment", "Тональность"), ("aspects", "Категории")]):
    h = pd.DataFrame(model2.history[task]).set_index("epoch")
    ax.plot(h.index, h["loss"], marker="o", label="потери (обучение)")
    if "val_f1" in h:
        ax.plot(h.index, h["val_f1"], marker="o", label="F1 (валидация)")
    ax.set_title(title); ax.set_xlabel("эпоха"); ax.legend()
plt.tight_layout()

## 5. Сравнение на тесте

In [ ]:
metrics2 = baseline.evaluate(model2, test_df)
print(baseline.format_report(metrics2))

In [ ]:
from aci_ml.schema import ASPECT_NAMES

def row(m):
    r = {"F1 тональность": m["f1_sentiment"], "F1 категории": m["f1_categories"]}
    r.update({ASPECT_NAMES[a]: v for a, v in m["f1_per_aspect"].items()})
    r["с на 1000 отзывов"] = m["seconds_per_1000"]
    return r

comparison = pd.DataFrame({"TF-IDF + LogReg": row(metrics1), "rubert-tiny2": row(metrics2)}).T.round(3)
comparison

**Для отчёта:** укажите размер обучающей и тестовой выборок, источник данных, настройки (эпохи, LR) и эту таблицу. Если цель F1 не достигнута — это тоже результат: объясните по разбору ошибок ниже, чего не хватает (данных, редких классов, длинных отзывов).

## 6. Где модели ошибаются и в чём расходятся

In [ ]:
texts = test_df["text"].tolist()
p1 = [r["sentiment"] for r in baseline.predict(model1, texts)]
p2 = [r["sentiment"] for r in baseline.predict(model2, texts)]
review = test_df[["text", "sentiment"]].assign(tfidf=p1, rubert=p2)
pd.set_option("display.max_colwidth", 120)

print("rubert-tiny2 ошибся:")
display(review[review["rubert"] != review["sentiment"]].head(15))
print("rubert-tiny2 прав, а TF-IDF — нет:")
display(review[(review["rubert"] == review["sentiment"]) & (review["tfidf"] != review["sentiment"])].head(15))

Трудные примеры — те же, на которых ошибается словарная модель сайта:

In [ ]:
examples = [
    "Не сказать, что вкусно",
    "Официант принёс холодный суп и даже не извинился",
    "Неплохо, но дороговато",
    "Ну конечно, «быстрая» доставка — всего полтора часа",
    "Всё прекрасно, только музыка громковата",
]
r1, r2 = baseline.predict(model1, examples), baseline.predict(model2, examples)
pd.DataFrame({"отзыв": examples,
              "TF-IDF": [f"{r['sentiment']}: {', '.join(r['categories'])}" for r in r1],
              "rubert-tiny2": [f"{r['sentiment']}: {', '.join(r['categories'])}" for r in r2]})

## 7. Сохранить модель

Модель сохранится в `ml/models/rubert_tiny2/` (около 120 МБ). В git её не добавляем — она слишком большая. В Colab ячейка ещё и скачает архив на компьютер.

Чтобы API отвечал этой моделью: `ACI_MODEL=models/rubert_tiny2 uvicorn aci_ml.api:app`.

In [ ]:
import shutil
out = ML / "models" / "rubert_tiny2"
model2.save(out)
print("Сохранено:", out)

if "google.colab" in sys.modules:
    from google.colab import files
    files.download(shutil.make_archive(str(out), "zip", out))